# NCF candidate-generation baseline

A static implicit-feedback neural collaborative filtering (NCF) baseline
trained on session clicks.

- Positives are session clicks from `training_negative_groups`; article
  history exposures are no longer treated as positives.
- The model contains two `nn.Embedding` blocks: one for users and one for
  articles. Their 64-d vectors are concatenated into a 128-d vector and passed
  through an MLP with shape `128 -> 64 -> 32 -> 1`.
- There is no dot product in the score.
- Training uses the `LOSS` objective (BPR or BCE) with one stored session
  negative sampled per positive each epoch.
- Validation candidates exclude the request's full causal history and use the
  eligible catalogue at the request timestamp.

The notebook runs the full pipeline by default. Set `FULL_RUN = False` in the
setup cell to execute a small smoke run. Full MLP scoring is heavier than GMF
because the MLP is evaluated for every request-article pair.

## 1. Setup

In [ ]:
from __future__ import annotations

import json
import sys
import time
from datetime import datetime, timezone
from pathlib import Path

import numpy as np
import polars as pl
import torch
import torch.nn.functional as F
from torch import nn

ROOT = next(
    p for p in [Path.cwd(), *Path.cwd().parents] if (p / "run_all.py").is_file()
)
CG_DIR = ROOT / "5_Models" / "5_01_Candidate_Generation"
NEURAL_DIR = CG_DIR / "neural_models"
sys.path.insert(0, str(CG_DIR))
sys.path.insert(0, str(NEURAL_DIR))
sys.path.insert(0, str(ROOT / "src"))

from cf_models import load_context, resolve_device
from cg_result_writer import CandidateWriter
from neural_helpers import (
    EarlyStopper,
    chronological_cutoff,
    chronological_mask,
    max_clicks_per_impression,
    mean_slate_metrics,
    tuning_slates,
)

PROCESSED = ROOT / "1_Data_Preparation" / "processed"
FEATURES = ROOT / "4_Feature_Engineering" / "artifacts" / "prequential-all"
REPRESENTATIONS = ROOT / "3_Representations" / "artifacts"
ARTIFACTS = ROOT / "5_Models" / "5_01_Candidate_Generation" / "artifacts"
RESULTS = ROOT / "results"

TARGET_SPLIT = "validation"
FULL_RUN = False                # Toggle here: False runs a two-epoch smoke pipeline.
RUN_TAG = "ncf" if FULL_RUN else "ncf_smoke"
MODEL_NAME = "ncf"
ARTIFACT_DIR = ARTIFACTS / RUN_TAG
RECOMMENDATIONS_PATH = ARTIFACT_DIR / f"recommendations_{TARGET_SPLIT}_top500.parquet"
RESULT_PATH = RESULTS / f"candidate_generation_{MODEL_NAME}_{TARGET_SPLIT}.csv"
COMPARISON_PATH = RESULTS / f"candidate_generation_comparison_{TARGET_SPLIT}.csv"

M_VALUES = (50, 100, 200, 500)
CANDIDATE_CAP = max(M_VALUES)
EMBED_DIM = 128
MLP_HIDDEN = (128, 64, 32)
SCORE_PAIRS = 524_288
NUM_EPOCHS = 50 if FULL_RUN else 20
BATCH_SIZE = 32_768
LEARNING_RATE = 1e-3
WEIGHT_DECAY = 1e-6
SEED = 4222
LOSS = "bpr"  # Toggle here: "bpr" or "bce".
if LOSS not in {"bpr", "bce"}:
    raise ValueError(f"Unknown loss: {LOSS}")
PATIENCE = 3 if FULL_RUN else 1
TUNE_QUANTILE = 0.9
TUNE_CANDIDATES = PROCESSED / "candidates_train.parquet"
SMOKE_GROUPS = 300_000
SMOKE_REQUESTS = 6_000
SMOKE_TUNE_IMPRESSIONS = 1_000
DEVICE = resolve_device("auto")

np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)
print("Repository:", ROOT)
print("Mode:", "full" if FULL_RUN else "smoke")
print("Device:", DEVICE)
print("Torch:", torch.__version__)

## 2. Data ingestion and model-related processing

### Session clicks and stored session negatives

Positives come from `training_negative_groups`: each row is one clicked article
from a training request, so only clicks in that session are treated as positive.
History-only exposures are no longer positives. The stored `negative_news_codes`
are observed non-clicks from the same slate; one is sampled per group in each
epoch. Article codes are mapped to `article_row` through `article_index.parquet`.

In [ ]:
context = load_context(ROOT)

In [ ]:
def load_training_groups():
    parts = sorted((FEATURES / "training_negative_groups").glob("*.parquet"))
    if not parts:
        raise FileNotFoundError("No training_negative_groups partitions")
    groups = pl.read_parquet(parts)
    if groups["negative_news_codes"].list.len().min() < 1:
        raise ValueError("A training group has no negative article")
    if not FULL_RUN and groups.height > SMOKE_GROUPS:
        groups = groups.sample(n=SMOKE_GROUPS, seed=SEED, shuffle=True)
    return groups

In [ ]:
article_index = pl.read_parquet(
    REPRESENTATIONS / "article_index.parquet",
    columns=["article_row", "news_code"],
).sort("article_row")

if article_index.height + 1 != context.n_items:
    raise ValueError("article_index does not align with context.n_items")

training_groups = load_training_groups().with_row_index("group_row")

positive_lookup = article_index.select(
    pl.col("news_code"),
    pl.col("article_row").alias("pos_row"),
)

training_groups = training_groups.join(
    positive_lookup,
    left_on="positive_news_code",
    right_on="news_code",
    how="left",
)

if training_groups["pos_row"].null_count():
    raise ValueError("A positive article is missing from article_index")

negative_lookup = article_index.select(
    pl.col("news_code").alias("negative_news_code"),
    pl.col("article_row").alias("neg_row"),
)

negative_long = (
    training_groups.select("group_row", "negative_news_codes")
    .explode("negative_news_codes")
    .rename({"negative_news_codes": "negative_news_code"})
    .with_columns(pl.int_range(pl.len()).over("group_row").alias("neg_pos"))
    .join(negative_lookup, on="negative_news_code", how="left")
    .sort(["group_row", "neg_pos"])
)

if negative_long["neg_row"].null_count():
    raise ValueError("A negative article is missing from article_index")

n_groups = training_groups.height

NEG_MAX = int(training_groups["negative_news_codes"].list.len().max())

NEG_MATRIX = np.zeros((n_groups, NEG_MAX), dtype=np.int64)

group_rows = negative_long["group_row"].to_numpy().astype(np.int64)

negative_positions = negative_long["neg_pos"].to_numpy().astype(np.int64)

negative_rows = negative_long["neg_row"].to_numpy().astype(np.int64)

NEG_MATRIX[group_rows, negative_positions] = negative_rows

NEG_LEN = np.bincount(group_rows, minlength=n_groups).astype(np.int64)

if (NEG_LEN < 1).any():
    raise ValueError("At least one training group has no negative")

train_profiles = pl.read_parquet(
    sorted((FEATURES / "request_profiles_train").glob("*.parquet")),
    columns=["impression_uid", "user_code", "timestamp"],
)

if train_profiles["impression_uid"].n_unique() != train_profiles.height:
    raise ValueError("Duplicate impression_uid in training profiles")

training_groups = (
    training_groups.join(train_profiles, on="impression_uid", how="left")
    .sort("group_row")
)

if training_groups["user_code"].null_count():
    raise ValueError("Missing training profile for a group")

if not np.array_equal(
    training_groups["group_row"].to_numpy().astype(np.int64),
    np.arange(n_groups, dtype=np.int64),
):
    raise ValueError("Training group order changed during joins")

train_users = training_groups["user_code"].to_numpy().astype(np.int64)

train_items = training_groups["pos_row"].to_numpy().astype(np.int64)

if train_users.min() < 0 or train_users.max() >= context.n_users:
    raise ValueError("Training user_code is outside the matrix")

if train_items.min() < 1 or train_items.max() >= context.n_items:
    raise ValueError("Training positive article row is outside the catalogue")

if negative_rows.min() < 1 or negative_rows.max() >= context.n_items:
    raise ValueError("Training negative article row is outside the catalogue")

print(f"Users: {context.n_users:,}")

print(f"Articles: {context.n_items:,}")

print(f"Training groups (session clicks): {n_groups:,}")

print(f"Positive pairs: {len(train_users):,}")

print(f"Negative memberships: {negative_rows.size:,}")

print(f"Max stored negatives per group: {NEG_MAX}")

TUNE_NDCG_K = max_clicks_per_impression(TUNE_CANDIDATES)

cutoff_ns = chronological_cutoff(
    training_groups["timestamp"].to_numpy(), TUNE_QUANTILE
)

fit_mask = chronological_mask(
    training_groups["timestamp"].to_numpy(), cutoff_ns
)

fit_idx = np.flatnonzero(fit_mask)

FIT_USERS = train_users[fit_idx]

FIT_ITEMS = train_items[fit_idx]

FIT_NEG_MATRIX = NEG_MATRIX[fit_idx]

FIT_NEG_LEN = NEG_LEN[fit_idx]

tune = tuning_slates(
    TUNE_CANDIDATES,
    cutoff_ns,
    article_index,
    train_profiles.select("impression_uid", "user_code"),
    max_impressions=None if FULL_RUN else SMOKE_TUNE_IMPRESSIONS,
    seed=SEED + 1,
)

if tune["article_row"].null_count() or tune["user_code"].null_count():
    raise ValueError("Tuning rows have a missing article or user")

TUNE_UIDS = tune["impression_uid"].to_numpy()

TUNE_USERS = tune["user_code"].to_numpy().astype(np.int64)

TUNE_ITEMS = tune["article_row"].to_numpy().astype(np.int64)

TUNE_CLICKED = tune["clicked"].to_numpy().astype(np.float32)

print(f"Chronological fit groups: {len(fit_idx):,} / {n_groups:,}")

print(f"Tuning cutoff: {np.datetime64(cutoff_ns, 'ns')}")

print(f"Tuning rows: {tune.height:,}")

print(
    "Tuning NDCG k (max training clicks per impression): "
    f"{TUNE_NDCG_K}"
)

## 3. Setting up the model

### Two embeddings, concatenation and an MLP

`NCF` has exactly two embedding tables. The user and article vectors are
concatenated into a 128-d vector and passed through `128 -> 64 -> 32 -> 1` with
ReLU activations. The scalar MLP output is the ranking score. BPR compares the
MLP score of a session-click positive pair with that of one stored exposed
non-click sampled from the same session. No false-negative filtering is applied.

In [ ]:
class NCF(nn.Module):
    def __init__(self, n_users, n_items, embed_dim, hidden_dims):
        super().__init__()
        self.user_embedding = nn.Embedding(n_users, embed_dim)
        self.item_embedding = nn.Embedding(n_items, embed_dim)
        nn.init.normal_(self.user_embedding.weight, std=0.01)
        nn.init.normal_(self.item_embedding.weight, std=0.01)
        with torch.no_grad():
            self.item_embedding.weight[0].zero_()

        layers = []
        in_dim = 2 * embed_dim
        for hidden_dim in hidden_dims:
            layers.append(nn.Linear(in_dim, hidden_dim))
            layers.append(nn.ReLU())
            in_dim = hidden_dim
        layers.append(nn.Linear(in_dim, 1))
        self.mlp = nn.Sequential(*layers)

    def score_embeddings(self, user_vectors, item_vectors):
        return self.mlp(
            torch.cat([user_vectors, item_vectors], dim=-1)
        ).squeeze(-1)

    def forward(self, user_ids, item_ids):
        return self.score_embeddings(
            self.user_embedding(user_ids),
            self.item_embedding(item_ids),
        )

## 4. Training the model

### BPR/BCE training, chronological tuning and refit

In [ ]:
def train_epoch(model, optimizer, user_ids, item_ids, neg_matrix, neg_len, rng):
    model.train()
    order = rng.permutation(len(user_ids))
    total = 0.0
    for start in range(0, len(order), BATCH_SIZE):
        batch = order[start : start + BATCH_SIZE]
        users = torch.as_tensor(
            user_ids[batch], dtype=torch.long, device=DEVICE
        )
        positives = torch.as_tensor(
            item_ids[batch], dtype=torch.long, device=DEVICE
        )
        choice = rng.integers(0, neg_len[batch])
        negatives = torch.as_tensor(
            neg_matrix[batch, choice], dtype=torch.long, device=DEVICE
        )
        optimizer.zero_grad(set_to_none=True)
        positive_scores = model(users, positives)
        negative_scores = model(users, negatives)
        if LOSS == "bpr":
            loss = -F.logsigmoid(positive_scores - negative_scores).mean()
        else:
            scores = torch.cat([positive_scores, negative_scores])
            labels = torch.cat(
                [
                    torch.ones_like(positive_scores),
                    torch.zeros_like(negative_scores),
                ]
            )
            loss = F.binary_cross_entropy_with_logits(scores, labels)
        loss.backward()
        optimizer.step()
        total += float(loss.item()) * len(batch)
    return total / len(order)


In [ ]:
def tune_ndcg(model):
    model.eval()
    scores = np.empty(len(TUNE_CLICKED), dtype=np.float32)
    with torch.no_grad():
        for start in range(0, len(TUNE_CLICKED), 200_000):
            end = min(start + 200_000, len(TUNE_CLICKED))
            users = torch.as_tensor(
                TUNE_USERS[start:end], dtype=torch.long, device=DEVICE
            )
            items = torch.as_tensor(
                TUNE_ITEMS[start:end], dtype=torch.long, device=DEVICE
            )
            scores[start:end] = model(users, items).cpu().numpy()
    _, ndcg = mean_slate_metrics(
        TUNE_UIDS, TUNE_CLICKED, scores, TUNE_NDCG_K
    )
    return ndcg

In [ ]:
model = NCF(
    context.n_users, context.n_items, EMBED_DIM, MLP_HIDDEN
).to(DEVICE)

optimizer = torch.optim.AdamW(
    model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY
)

rng = np.random.default_rng(SEED + 1)

training_log = []

stopper = EarlyStopper(patience=PATIENCE)

for epoch in range(1, NUM_EPOCHS + 1):
    started = time.time()
    loss = train_epoch(
        model,
        optimizer,
        FIT_USERS,
        FIT_ITEMS,
        FIT_NEG_MATRIX,
        FIT_NEG_LEN,
        rng,
    )
    ndcg = tune_ndcg(model)
    elapsed = time.time() - started
    training_log.append(
        {
            "phase": "tune",
            "epoch": epoch,
            "train_loss": loss,
            "tune_ndcg": ndcg,
            "ndcg_k": TUNE_NDCG_K,
            "seconds": elapsed,
        }
    )
    print(
        f"epoch {epoch:02d} loss={loss:.6f} "
        f"tune_ndcg@{TUNE_NDCG_K}={ndcg:.5f} ({elapsed:.1f}s)"
    )
    if stopper.step(ndcg, model, epoch):
        print("Early stopping")
        break

stopper.restore(model)

print(
    f"Best epoch: {stopper.best_epoch} "
    f"(tune_ndcg@{TUNE_NDCG_K}={stopper.best_score:.5f})"
)

final_model = NCF(
    context.n_users, context.n_items, EMBED_DIM, MLP_HIDDEN
).to(DEVICE)

optimizer = torch.optim.AdamW(
    final_model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY
)

rng = np.random.default_rng(SEED + 2)

for epoch in range(1, stopper.best_epoch + 1):
    started = time.time()
    loss = train_epoch(
        final_model,
        optimizer,
        train_users,
        train_items,
        NEG_MATRIX,
        NEG_LEN,
        rng,
    )
    elapsed = time.time() - started
    training_log.append(
        {
            "phase": "refit",
            "epoch": epoch,
            "train_loss": loss,
            "tune_ndcg": float("nan"),
            "ndcg_k": TUNE_NDCG_K,
            "seconds": elapsed,
        }
    )
    print(f"refit epoch {epoch:02d} loss={loss:.6f} ({elapsed:.1f}s)")

model = final_model

torch.save(
    {
        "state_dict": model.state_dict(),
        "config": {
            "n_users": context.n_users,
            "n_items": context.n_items,
            "embed_dim": EMBED_DIM,
            "mlp_hidden": list(MLP_HIDDEN),
            "objective": LOSS,
            "num_negatives": 1,
            "max_epochs": NUM_EPOCHS,
            "best_epoch": stopper.best_epoch,
            "epochs": stopper.best_epoch,
            "tune_ndcg": stopper.best_score,
            "ndcg_k": TUNE_NDCG_K,
            "batch_size": BATCH_SIZE,
            "learning_rate": LEARNING_RATE,
            "weight_decay": WEIGHT_DECAY,
            "seed": SEED,
            "training_data": "training_negative_groups",
            "training_groups": n_groups,
            "negative_max": NEG_MAX,
        },
    },
    ARTIFACT_DIR / "model.pt",
)

pl.DataFrame(training_log).write_csv(ARTIFACT_DIR / "training_log.csv")

print("Saved model:", ARTIFACT_DIR / "model.pt")

## 5. Results

### Candidate helpers

These helpers keep the candidate format, eligibility policy, history exclusion
and popularity tie-breaking consistent with the other candidate generators.

In [ ]:
def popularity_select(order, n_eligible, excluded=None, k=CANDIDATE_CAP):
    excluded_set = set() if excluded is None else set(map(int, excluded))
    selected = []
    for position in order:
        position = int(position)
        if position >= n_eligible or position in excluded_set:
            continue
        selected.append(position)
        if len(selected) == k:
            return np.asarray(selected, dtype=np.int64)
    raise ValueError(
        f"only {len(selected)} popular eligible articles; cannot select {k}"
    )

In [ ]:
def popularity_fill(available, popularity, news_code, count):
    pool = np.flatnonzero(available)
    if pool.size < count:
        raise ValueError(
            f"only {pool.size} eligible articles remain; cannot fill {count}"
        )
    order = np.lexsort((news_code[pool], -popularity[pool]))
    return pool[order[:count]]

In [ ]:
def select_top_k(
    scores, popularity, news_code, k, excluded=None, fallback_order=None
):
    scores = np.asarray(scores, dtype=np.float32)
    popularity = np.asarray(popularity)
    news_code = np.asarray(news_code)
    if scores.ndim != 1:
        raise ValueError("scores must be one-dimensional")
    if popularity.shape != scores.shape or news_code.shape != scores.shape:
        raise ValueError("score, popularity and news_code shapes must match")
    if k < 1:
        raise ValueError("k must be positive")
    n = scores.size
    if n < k:
        raise ValueError(f"only {n} eligible articles; cannot select {k}")
    if fallback_order is not None:
        return popularity_select(
            fallback_order, n, excluded=excluded, k=k
        )

    available = np.ones(n, dtype=bool)
    if excluded is not None and len(excluded):
        positions = np.asarray(excluded, dtype=np.int64)
        if positions.min() < 0 or positions.max() >= n:
            raise ValueError("excluded position outside the eligible array")
        available[positions] = False
        scores = scores.copy()
        scores[positions] = -np.inf

    finite = np.isfinite(scores)
    n_finite = int(np.count_nonzero(finite))
    if n_finite == 0:
        return popularity_fill(
            available, popularity, news_code, k
        )

    take = min(k, n_finite)
    if take < n_finite:
        frontier = np.argpartition(-scores, take - 1)[:take]
        threshold = scores[frontier].min()
    else:
        threshold = scores[finite].min()

    above = np.flatnonzero(scores > threshold)
    need = take - above.size
    ties = np.flatnonzero(scores == threshold)
    if need > 0:
        if ties.size > need:
            tie_order = np.lexsort(
                (news_code[ties], -popularity[ties])
            )
            ties = ties[tie_order[:need]]
        chosen_positions = np.r_[above, ties]
    else:
        chosen_positions = above

    order = np.lexsort(
        (
            news_code[chosen_positions],
            -popularity[chosen_positions],
            -scores[chosen_positions],
        )
    )
    selected = chosen_positions[order].astype(np.int64)

    if selected.size < k:
        remaining = available.copy()
        remaining[selected] = False
        fill = popularity_fill(
            remaining, popularity, news_code, k - selected.size
        )
        selected = np.r_[selected, fill]

    return selected

### Score the eligible catalogue and write Top-500 candidates

Each request uses the static embedding of its `user_code`. Articles not yet
available at the request timestamp are removed, every article in the request's
`full_history_rows` is excluded, and empty-history requests fall back to
popularity order. Ties are broken by popularity and then `news_code`.

Because the MLP is not a bilinear dot product, scores are evaluated in bounded
pair chunks rather than one matrix multiply. Full validation scoring is
materially heavier than a dot-product model, so smoke mode is useful before a full run.

In [ ]:
def load_profiles(split):
    parts = sorted((FEATURES / f"request_profiles_{split}").glob("*.parquet"))
    if not parts:
        raise FileNotFoundError(f"No request profiles for {split}")
    frame = pl.read_parquet(
        parts,
        columns=[
            "impression_uid",
            "user_code",
            "timestamp",
            "full_history_rows",
        ],
    )
    if frame["impression_uid"].n_unique() != frame.height:
        raise ValueError("Duplicate impression_uid in request profiles")
    if frame["full_history_rows"].null_count():
        raise ValueError("Null full_history_rows in request profiles")
    return frame.sort("timestamp", maintain_order=True)

In [ ]:
def score_batch(model, user_vectors, item_vectors, max_pairs=SCORE_PAIRS):
    n_users, n_items = user_vectors.shape[0], item_vectors.shape[0]
    scores = np.empty((n_users, n_items), dtype=np.float32)
    if n_users == 0 or n_items == 0:
        return scores

    users_t = torch.from_numpy(user_vectors).to(DEVICE)
    items_t = torch.from_numpy(item_vectors).to(DEVICE)
    items_per_chunk = min(n_items, max_pairs)
    users_per_chunk = max(1, max_pairs // items_per_chunk)
    with torch.no_grad():
        for u0 in range(0, n_users, users_per_chunk):
            u1 = min(u0 + users_per_chunk, n_users)
            user_block = users_t[u0:u1]
            for i0 in range(0, n_items, items_per_chunk):
                i1 = min(i0 + items_per_chunk, n_items)
                item_block = items_t[i0:i1]
                rows = u1 - u0
                cols = i1 - i0
                user_expand = user_block[:, None, :].expand(
                    rows, cols, EMBED_DIM
                )
                item_expand = item_block[None, :, :].expand(
                    rows, cols, EMBED_DIM
                )
                scores[u0:u1, i0:i1] = model.score_embeddings(
                    user_expand, item_expand
                ).cpu().numpy()
    return scores

In [ ]:
catalogue = pl.DataFrame(
    {
        "article_row": np.arange(1, context.n_items, dtype=np.int32),
        "news_code": context.news_code[1:],
        "available_from": context.available_from[1:],
        "popularity": context.popularity[1:],
    }
)

ordered = catalogue.sort(["available_from", "news_code"])

article_rows = ordered["article_row"].to_numpy().astype(np.int64)

news_sorted = ordered["news_code"].to_numpy().astype(np.int64)

available_sorted = ordered["available_from"].to_numpy().astype(
    "datetime64[ns]"
)

popularity_sorted = ordered["popularity"].to_numpy().astype(np.float32)

pos_in_sorted = np.full(context.n_items, -1, dtype=np.int64)

pos_in_sorted[article_rows] = np.arange(article_rows.size, dtype=np.int64)

global_popularity_order = np.lexsort((news_sorted, -popularity_sorted))

if not np.all(available_sorted[1:] >= available_sorted[:-1]):
    raise ValueError("Availability order is not non-decreasing")

profiles = load_profiles(TARGET_SPLIT)

impression_uids = profiles["impression_uid"].to_numpy()

target_users = profiles["user_code"].to_numpy().astype(np.int64)

timestamps = profiles["timestamp"].to_numpy().astype("datetime64[ns]")

history_rows = profiles["full_history_rows"].to_list()

if target_users.min() < 0 or target_users.max() >= context.n_users:
    raise ValueError("Profile user_code is outside the matrix")

has_history = np.fromiter(
    (len(rows) > 0 for rows in history_rows),
    dtype=bool,
    count=len(history_rows),
)

starts = np.flatnonzero(np.r_[True, timestamps[1:] != timestamps[:-1]])

ends = np.r_[starts[1:], len(timestamps)]

batches = list(zip(starts, ends))

if not FULL_RUN:
    batch_rng = np.random.default_rng(SEED)
    batch_order = batch_rng.permutation(len(batches))
    selected_batches = []
    total = 0
    for index in batch_order:
        start, end = batches[index]
        selected_batches.append((start, end))
        total += end - start
        if total >= SMOKE_REQUESTS:
            break
    batches = sorted(selected_batches)

processed_requests = int(sum(end - start for start, end in batches))

print(f"Scoring requests: {processed_requests:,} / {len(timestamps):,}")

model.eval()

user_emb = model.user_embedding.weight.detach().cpu().numpy()[target_users]

item_emb = model.item_embedding.weight.detach().cpu().numpy()[article_rows]

cutoff_values = np.arange(1, CANDIDATE_CAP + 1, dtype=np.int16)

writer = CandidateWriter(RECOMMENDATIONS_PATH)

processed_uids = []

started = time.time()

try:
    for batch_no, (start, end) in enumerate(batches, 1):
        timestamp = timestamps[start]
        n_eligible = int(
            np.searchsorted(available_sorted, timestamp, side="right")
        )
        if n_eligible < CANDIDATE_CAP:
            raise ValueError(
                f"Only {n_eligible} eligible articles at {timestamp}"
            )
        scores = score_batch(model, user_emb[start:end], item_emb[:n_eligible])
        popularity_batch = popularity_sorted[:n_eligible]
        news_batch = news_sorted[:n_eligible]
        selected_batch = np.empty(
            (end - start, CANDIDATE_CAP), dtype=np.int64
        )
        for local, profile_row in enumerate(range(start, end)):
            rows = history_rows[profile_row]
            if rows:
                excluded = pos_in_sorted[
                    np.asarray(rows, dtype=np.int64)
                ]
                excluded = excluded[
                    (excluded >= 0) & (excluded < n_eligible)
                ]
            else:
                excluded = np.empty(0, dtype=np.int64)
            fallback = (
                None
                if has_history[profile_row]
                else global_popularity_order
            )
            selected_batch[local] = select_top_k(
                scores[local],
                popularity_batch,
                news_batch,
                CANDIDATE_CAP,
                excluded=excluded,
                fallback_order=fallback,
            )

        writer.add(
            np.repeat(impression_uids[start:end], CANDIDATE_CAP),
            np.tile(cutoff_values, end - start),
            news_batch[selected_batch.reshape(-1)],
        )
        processed_uids.append(impression_uids[start:end].copy())
        if batch_no % 1000 == 0 or batch_no == len(batches):
            print(
                f"Scored {batch_no:,}/{len(batches):,} timestamp batches; "
                f"{writer.rows:,} rows",
                flush=True,
            )
finally:
    writer.close()

elapsed = time.time() - started

expected_rows = processed_requests * CANDIDATE_CAP

actual_rows = (
    pl.scan_parquet(RECOMMENDATIONS_PATH)
    .select(pl.len())
    .collect()
    .item()
)

if actual_rows != expected_rows:
    raise ValueError(
        f"Expected {expected_rows:,} rows, found {actual_rows:,}"
    )

if not FULL_RUN:
    check = pl.read_parquet(RECOMMENDATIONS_PATH)
    if check.select(pl.col("cutoff").n_unique()).item() != CANDIDATE_CAP:
        raise ValueError("Smoke output is missing cutoff values")
    if check.group_by("impression_uid").agg(
        pl.col("news_code").n_unique().alias("unique_news")
    ).filter(pl.col("unique_news") != CANDIDATE_CAP).height:
        raise ValueError("Smoke output contains duplicate candidates")

subset_uids = (
    np.concatenate(processed_uids)
    if processed_uids
    else np.empty(0, dtype=np.int64)
)

print(f"Wrote {actual_rows:,} candidate rows in {elapsed:.1f}s")

print("Candidates:", RECOMMENDATIONS_PATH)

### Manifest, evaluation and comparison

Full runs use the shared `CandidateEvaluator`. Smoke runs score only a request
subset, so they compute subset recall with the same metric columns and write
`smoke_recall_<split>.csv` instead of calling the shared evaluator.

In [ ]:
def smoke_recall(recommendations_path, model_name, impression_uids):
    uid_frame = pl.DataFrame(
        {"impression_uid": np.asarray(impression_uids, dtype=np.int64)}
    )
    label_parts = sorted(
        (FEATURES / f"labels_{TARGET_SPLIT}").glob("*.parquet")
    )
    if not label_parts:
        raise FileNotFoundError(f"No labels for {TARGET_SPLIT}")
    labels = pl.read_parquet(
        label_parts,
        columns=["impression_uid", "news_code", "clicked"],
    ).join(uid_frame, on="impression_uid", how="inner")
    positives = labels.filter(pl.col("clicked") == 1).select(
        "impression_uid", "news_code"
    )
    relevant = positives.group_by("impression_uid").agg(
        pl.len().alias("relevant")
    )
    base = uid_frame.join(
        relevant, on="impression_uid", how="left"
    ).with_columns(pl.col("relevant").fill_null(0).cast(pl.Int64))
    candidates = pl.read_parquet(
        recommendations_path,
        columns=["impression_uid", "news_code", "cutoff"],
    )
    rows = []
    for m in M_VALUES:
        selected = candidates.filter(pl.col("cutoff") <= m)
        hits = (
            positives.join(
                selected.select("impression_uid", "news_code"),
                on=["impression_uid", "news_code"],
                how="inner",
            )
            .group_by("impression_uid")
            .agg(pl.len().alias("hits"))
        )
        per_impression = base.join(
            hits, on="impression_uid", how="left"
        ).with_columns(pl.col("hits").fill_null(0).cast(pl.Int64))
        relevant_clicks = int(per_impression["relevant"].sum())
        retrieved_relevant = int(per_impression["hits"].sum())
        with_relevant = per_impression.filter(pl.col("relevant") > 0)
        recall_impression_mean = (
            float(
                (
                    with_relevant["hits"] / with_relevant["relevant"]
                ).mean()
            )
            if with_relevant.height
            else float("nan")
        )
        hit_rate = (
            float((per_impression["hits"] > 0).sum() / per_impression.height)
            if per_impression.height
            else float("nan")
        )
        rows.append(
            {
                "model": model_name,
                "split": TARGET_SPLIT,
                "m": int(m),
                "recall_pooled": (
                    retrieved_relevant / relevant_clicks
                    if relevant_clicks
                    else float("nan")
                ),
                "recall_impression_mean": recall_impression_mean,
                "hit_rate": hit_rate,
                "impressions": int(per_impression.height),
                "relevant_clicks": relevant_clicks,
                "retrieved_relevant": retrieved_relevant,
                "mean_candidates": float(min(m, CANDIDATE_CAP)),
            }
        )
    return pl.DataFrame(rows)

In [ ]:
smoke_recall_path = None
if not FULL_RUN:
    smoke_evaluation = smoke_recall(
        RECOMMENDATIONS_PATH, MODEL_NAME, subset_uids
    )
    smoke_recall_path = ARTIFACT_DIR / f"smoke_recall_{TARGET_SPLIT}.csv"
    smoke_evaluation.write_csv(smoke_recall_path)
    print(smoke_evaluation.sort("m"))
    print("Smoke recall:", smoke_recall_path)

In [ ]:
manifest = {
    "model": MODEL_NAME,
    "score": "MLP(concat(user_embedding, item_embedding))",
    "embedding_blocks": ["user_embedding", "item_embedding"],
    "mlp": [2 * EMBED_DIM, *MLP_HIDDEN, 1],
    "activation": "ReLU",
    "objective": (
        f"{LOSS.upper()} on session clicks with one stored session "
        "negative per positive"
    ),
    "training_data": (
        "4_Feature_Engineering/artifacts/prequential-all/"
        "training_negative_groups"
    ),
    "positive_policy": "clicked articles in the request session only",
    "negative_policy": (
        "one exposed non-click sampled from the same session per epoch"
    ),
    "fit_split": "train",
    "target_split": TARGET_SPLIT,
    "candidate_cap": CANDIDATE_CAP,
    "candidate_format": ["impression_uid", "cutoff", "news_code"],
    "eligibility": "available_from <= timestamp",
    "history_policy": "exclude every article in full_history_rows",
    "fallback": "popularity order for empty-history requests or short lists",
    "mode": "full" if FULL_RUN else "smoke",
    "seed": SEED,
    "device": DEVICE,
    "torch_version": torch.__version__,
    "embed_dim": EMBED_DIM,
    "mlp_hidden": list(MLP_HIDDEN),
    "epochs": int(stopper.best_epoch),
    "max_epochs": NUM_EPOCHS,
    "best_epoch": int(stopper.best_epoch),
    "tune_ndcg": float(stopper.best_score),
    "ndcg_k": int(TUNE_NDCG_K),
    "batch_size": BATCH_SIZE,
    "train_pairs": int(len(train_users)),
    "training_groups": int(n_groups),
    "negative_memberships": int(negative_rows.size),
    "negative_max": int(NEG_MAX),
    "target_requests": int(processed_requests),
    "recommendation_rows": int(actual_rows),
    "recommendations_file": RECOMMENDATIONS_PATH.name,
    "smoke_recall_file": (
        smoke_recall_path.name if smoke_recall_path is not None else None
    ),
    "model_file": "model.pt",
    "created_at": datetime.now(timezone.utc).isoformat(),
}
manifest_path = ARTIFACT_DIR / f"manifest_{TARGET_SPLIT}.json"
manifest_path.write_text(json.dumps(manifest, indent=2) + "\n")
print("Manifest:", manifest_path)

In [ ]:
if FULL_RUN:
    from cg_evaluation import CandidateEvaluator

    evaluator = CandidateEvaluator(root=ROOT, split=TARGET_SPLIT)
    evaluation = evaluator.evaluate(
        RECOMMENDATIONS_PATH,
        model_name=MODEL_NAME,
        m_values=M_VALUES,
    )
    evaluation.write_csv(RESULT_PATH)

    if COMPARISON_PATH.exists():
        existing = pl.read_csv(COMPARISON_PATH).filter(
            pl.col("model") != MODEL_NAME
        )
        common = [
            column
            for column in existing.columns
            if column in evaluation.columns
        ]
        comparison = pl.concat(
            [existing.select(common), evaluation.select(common)],
            how="vertical_relaxed",
        )
    else:
        comparison = evaluation
    comparison.write_csv(COMPARISON_PATH)
    print(evaluation.sort("m"))
    print("Results:", RESULT_PATH)
else:
    print(
        "Smoke run: shared evaluator skipped; subset recall written."
    )

## Notes and limitations

- The user representation is a static `user_code` embedding, not a causal
  request profile. Full `full_history_rows` is used only for exclusion and
  fallback.
- Positives are the clicked articles of each training request (session clicks);
  history-only exposures are not treated as positives.
- Negatives are observed non-clicks from the same slate and are already sampled
  to at most four per positive, so training does not see the full candidate set.
- Training uses a chronological tuning slice, early stopping and a final refit on all training groups.
- The MLP output is a ranking score, not a calibrated click probability.
- Full NCF scoring evaluates the MLP for every request-article pair and is much
  slower than a dot-product model.
- `available_from` is the project's observable-availability proxy, not a
  verified publication timestamp.